# Tip-Adapter & Tip-Adapter-F on CIFAR-10 with CLIP ViT-B/32
**Training-free vs fine-tuned cache adaptation of CLIP in the few-shot regime.**
Complete, reproducible pipeline for a Kaggle **Tesla T4 (16 GB)**.

---

## 1. Research question

> How effectively can Tip-Adapter and Tip-Adapter-F adapt a *frozen* pretrained CLIP
> to CIFAR-10 using only $k \in \{1,2,4,8,16\}$ labeled images per class?

We compare four approaches on identical support sets and the official test split:

| # | Method | What is learned | Trainable params (16-shot) |
|---|--------|-----------------|----------------------------|
| 1 | **Zero-shot CLIP** | nothing (prompt ensemble only) | 0 |
| 2 | **Linear Probe** | a linear head $W,b$ on frozen features | 5,130 |
| 3 | **Tip-Adapter** | nothing (non-parametric cache, tuned $\alpha,\beta$) | 0 |
| 4 | **Tip-Adapter-F** | the cache keys $K$ (gradient descent) | $N \times 512$ = 81,920 |

## 2. Design at a glance

```
CIFAR-10 (50,000 train / 10,000 test, 32x32)
     |  resize 224x224 bicubic + center crop + official CLIP normalization
     v
FROZEN CLIP ViT-B/32  (image + text encoders; requires_grad=False)
     |                                   |
     v                                   v
image banks (disk-cached)         10 classes x 3 templates
train [50000,512]  test [10000,512]   -> averaged normalized text embeddings [10,512]
     |
     |-- stratified splits per (shots, seed):  support [k/class]  +  validation [100/class]
     |
     +-- Zero-shot:      argmax( 100 * f_img . T^T )
     +-- Linear probe:   argmax( W f_img + b ),  cross-entropy on support
     +-- Tip-Adapter:    clip_logits + alpha * exp(-beta*(1 - f_img . K^T)) . V
     +-- Tip-Adapter-F:  same formula, but K is nn.Parameter, trained with AdamW
```

## 3. The Tip-Adapter cache model (both variants)

Support set of $N = 10k$ images with CLIP features $K \in \mathbb{R}^{N\times512}$
(L2-normalized) and one-hot labels $V \in \{0,1\}^{N\times10}$:

$$A(q, K) = \exp\big(-\beta\,(1 - qK^{\top})\big) \qquad \text{(affinity, in } (0,1])$$
$$\text{cache\_logits}(q) = A(q,K)\,V \qquad \text{final}(q) = \underbrace{100\, qT^{\top}}_{\text{CLIP logits}} + \; \alpha\, \text{cache\_logits}(q)$$

* **$\beta$ (affinity sharpness):** large $\beta$ -> only cache keys almost identical to the
  query contribute (nearest-neighbour behaviour); $\beta \to 0$ -> affinity ~ uniform.
* **$\alpha$ (cache weight):** global weight of the few-shot memory relative to CLIP's
  zero-shot logits; $\alpha = 0$ recovers pure zero-shot CLIP.

**Tip-Adapter-F** initializes $K$ from the support features and makes it the *only*
trainable tensor (`nn.Parameter`). $V$, the text embeddings, the logit scale and both
CLIP encoders are constants. Loss = cross-entropy on `final(q)` with the support
images themselves as queries (official protocol); AdamW; early stopping and checkpoint
selection on the **validation** set; after training $\alpha$ is re-searched on validation
with the learned keys frozen (official two-stage protocol).

## 4. Normalization strategy (exact)

* image and text embeddings from frozen CLIP: **L2-normalized**;
* text: per-template normalize -> average over 3 templates -> re-normalize;
* cache keys: initialized from **L2-normalized** support features, **not**
  re-normalized during training (matches the official implementation);
* queries: always L2-normalized frozen features, never updated;
* CLIP's learned logit scale ($\exp(\text{logit\_scale}) = 100$) is kept in the CLIP logits.

## 5. Experimental protocol (fairness rules)

1. Support sets: stratified, exactly $k$ per class, seeded by
   `seeds = [42, 123, 2026]`; **identical support images for all four methods**.
2. Validation: 100 images/class drawn from the train split **minus** the support set;
   every hyperparameter ($\alpha, \beta$, lr, weight decay, epochs) is chosen here.
3. Test: the official 10,000-image split, evaluated **once** per method, after all
   selection is frozen. No test labels are ever used for tuning, checkpoints, or
   early stopping.
4. Frozen CLIP embeddings are extracted once and cached to `feature_cache/*.pt`;
   all methods consume the same banks.
5. Report mean +/- std over the 3 seeds; per-run records, checkpoints, figures and
   config dumps are saved under `/kaggle/working`.

## 6. Ablations

* **A - Cache contribution:** zero-shot vs CLIP + cache -> what the few-shot memory adds.
* **B - Affinity function:** raw cosine vs $\exp(-\beta(1-\cos))$ with identical
  aggregation -> the value of the exponential kernel.
* **C - Fine-tuning effect:** fixed vs trainable cache keys -> what gradients buy.
* **D - Shots:** the main 1/2/4/8/16 sweep -> sample-efficiency of each method.

---

*Everything below is executable: it runs top-to-bottom on a T4 in ~20-30 minutes.*


## 7. Setup
`open_clip_torch` is the only package that is not preinstalled on Kaggle.
Everything else (torch, torchvision, pandas, scikit-learn, matplotlib) is already there.

In [ ]:
!pip install -q open_clip_torch

import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__,
      '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          '| VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))


## 8. Configuration
All constants, search grids and paths in one place. `MOCK` mode (env var `TIP_MOCK=1`)
replaces CLIP/CIFAR with synthetic features so the pipeline logic can be tested on CPU.

In [ ]:
# =============================================================================
# config.py -- Central configuration for the Tip-Adapter / CIFAR-10 study
#
# All hyperparameter search spaces live here so that the experiment is fully
# reproducible. Every search is performed on the VALIDATION split only; the
# official test set is touched exactly once per method, after selection.
# =============================================================================
import os

import torch

# --- Reproducibility ---------------------------------------------------------
SEEDS = [42, 123, 2026]            # seeds for support/validation sampling + init
SHOTS = [1, 2, 4, 8, 16]           # support images per class
NUM_CLASSES = 10
VAL_PER_CLASS = 100                # validation images per class (drawn from TRAIN only)

# --- CLIP --------------------------------------------------------------------
CLIP_ARCH = "ViT-B-32"             # open_clip name for OpenAI CLIP ViT-B/32
CLIP_PRETRAINED = "openai"         # original OpenAI pretrained weights
CLIP_HF_FALLBACK = "openai/clip-vit-base-patch32"   # used only if open_clip fails
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)     # official CLIP normalization
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
TEMPLATES = [
    "a photo of a {}.",
    "a picture of a {}.",
    "an image of a {}.",
]

# --- Performance (tuned for a Kaggle Tesla T4, 16 GB VRAM) --------------------
BATCH_SIZE = 256
NUM_WORKERS = 2
USE_AMP = True                     # fp16 autocast for frozen feature extraction
torch.backends.cudnn.benchmark = True

# --- Hyperparameter search spaces (tuned on VALIDATION only) ------------------
# Tip-Adapter training-free: alpha (cache weight) x beta (affinity sharpness)
ALPHA_GRID = [round(0.25 * i, 2) for i in range(25)]        # 0.00 .. 6.00
BETA_GRID = [2.0, 3.0, 4.0, 5.0, 5.86, 7.0, 8.0]            # 5.86 = paper default
# Linear probe: lr x weight decay, selected on validation
LP_LR_GRID = [1e-3, 3e-3, 1e-2, 3e-2]
LP_WD_GRID = [1e-4, 1e-2]
LP_EPOCHS = 300
LP_EVAL_EVERY = 25
# Tip-Adapter-F: lr x weight decay, early stopping on validation
TIPF_LR_GRID = [1e-3, 5e-3, 1e-2]
TIPF_WD_GRID = [1e-4, 1e-3]
TIPF_MAX_EPOCHS = 20
TIPF_PATIENCE = 5

# --- Paths --------------------------------------------------------------------
MOCK = os.environ.get("TIP_MOCK", "0") == "1"   # local CPU logic test with synthetic features
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_ROOT = "/tmp/cifar10" if os.path.isdir("/kaggle/working") else os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "feature_cache")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
FIG_DIR = os.path.join(WORK_DIR, "figures")
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")


def ensure_dirs():
    for d in (DATA_ROOT, CACHE_DIR, RESULTS_DIR, FIG_DIR, CKPT_DIR):
        os.makedirs(d, exist_ok=True)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 9. Data & few-shot splits
Official CIFAR-10 train/test splits; official CLIP preprocessing (bicubic resize to
224, center crop, CLIP mean/std). `make_few_shot_splits` samples the support set and
a disjoint 1,000-image validation set from the remaining *training* images, so
validation can never leak into support and test is never touched here.

In [ ]:
# =============================================================================
# data.py -- CIFAR-10 loading, official CLIP preprocessing, few-shot splits
#
# Split discipline (critical for a fair comparison):
#   * support set  : k images/class drawn from the official CIFAR-10 TRAIN split
#   * validation   : 100 images/class drawn from TRAIN \ support (never reused
#                    as support; used for ALL hyperparameter selection)
#   * test         : the official 10,000-image test split, evaluated ONCE per
#                    method, only after hyperparameters are frozen.
# =============================================================================
import numpy as np
import torch
from torch.utils.data import Dataset

# NOTE: this file is executed into a single shared namespace (by run_experiments.py
# locally, or by the notebook cells), so project-level names (CLIP_MEAN, ...,
# VAL_PER_CLASS, DATA_ROOT, MOCK) are taken from config.py defined earlier.


def clip_image_transform():
    """The official OpenAI CLIP preprocessing, applied to 32x32 CIFAR images:
       resize to 224 (bicubic) -> center crop 224 -> tensor -> CLIP mean/std.
       This is identical to open_clip's preprocessing for the OpenAI weights."""
    from torchvision import transforms as T
    return T.Compose([
        T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(CLIP_MEAN, CLIP_STD),
    ])


def load_datasets(root=DATA_ROOT, mock=MOCK):
    """Returns (train_dataset, test_dataset, train_targets, test_targets).
    Datasets are None in mock mode (features are synthesized instead)."""
    if mock:
        rng = np.random.RandomState(0)
        ytr = rng.randint(0, NUM_CLASSES, 2000).astype(np.int64)
        yte = rng.randint(0, NUM_CLASSES, 500).astype(np.int64)
        print("[data] MOCK mode: synthetic targets (2000 train / 500 test)")
        return None, None, ytr, yte
    import torchvision
    tf = clip_image_transform()
    train_ds = torchvision.datasets.CIFAR10(root=root, train=True, download=True, transform=tf)
    test_ds = torchvision.datasets.CIFAR10(root=root, train=False, download=True, transform=tf)
    return train_ds, test_ds, np.asarray(train_ds.targets), np.asarray(test_ds.targets)


def make_few_shot_splits(targets, shots, seed, val_per_class=VAL_PER_CLASS):
    """Stratified sampling with a per-seed RNG.

    1. support: exactly `shots` images per class (disjoint across classes).
    2. validation: exactly `val_per_class` images per class from the REMAINING
       train images, guaranteed disjoint from the support set.

    Returns (support_idx, val_idx), both sorted int64 arrays.
    """
    targets = np.asarray(targets)
    rng = np.random.RandomState(seed)
    classes = np.unique(targets)

    support_per_class = []
    for c in classes:
        pool = np.flatnonzero(targets == c)
        support_per_class.append(rng.choice(pool, size=shots, replace=False))
    support_idx = np.sort(np.concatenate(support_per_class)).astype(np.int64)

    rest = np.setdiff1d(np.arange(len(targets)), support_idx)
    val_per_class_idx = []
    for c in classes:
        pool = rest[targets[rest] == c]
        val_per_class_idx.append(rng.choice(pool, size=val_per_class, replace=False))
    val_idx = np.sort(np.concatenate(val_per_class_idx)).astype(np.int64)

    assert len(support_idx) == shots * len(classes)
    assert len(val_idx) == val_per_class * len(classes)
    assert not np.intersect1d(support_idx, val_idx).size, "support/val leakage"
    return support_idx, val_idx


## 10. Frozen CLIP + cached feature banks
The image encoder is run once over all 60,000 images (fp16 autocast on the T4) and
the normalized embeddings are cached to disk. The text side encodes 10 classes x 3
templates. Every downstream method operates on these banks only - the CLIP encoders
are deleted afterwards, which makes accidental CLIP fine-tuning impossible.

In [ ]:
# =============================================================================
# clip_features.py -- frozen CLIP loading, feature extraction, disk caching
#
# Normalization strategy (documented once, used everywhere):
#   * image features : forward through FROZEN image encoder, then L2-normalize
#   * text features  : per-template L2-normalize -> average over templates ->
#                      re-normalize per class (standard CLIP prompt ensembling)
#   * cache keys     : initialized from L2-normalized support image features;
#                      NOT re-normalized during fine-tuning (matches the
#                      official Tip-Adapter implementation; the keys drift
#                      freely so they can point to better class prototypes)
#   * queries        : always L2-normalized frozen CLIP features (never updated)
#
# All four methods consume the SAME cached feature banks, so the frozen CLIP
# model is only run over the images once per session.
# =============================================================================
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

try:
    from config import (BATCH_SIZE, CACHE_DIR, CLASSES, CLIP_ARCH, CLIP_HF_FALLBACK,
                        CLIP_PRETRAINED, DEVICE, MOCK, NUM_CLASSES, NUM_WORKERS,
                        TEMPLATES, USE_AMP)
except ImportError:
    pass  # shared-namespace mode (notebook): names already defined by config cell


class _HFTextImageWrapper(nn.Module):
    """Minimal adapter so a HuggingFace CLIPModel exposes the same
    encode_image / encode_text / logit_scale interface as open_clip."""

    def __init__(self, hf):
        super().__init__()
        self.hf = hf

    def encode_image(self, images):
        return self.hf.get_image_features(pixel_values=images)

    def encode_text(self, input_ids):
        return self.hf.get_text_features(input_ids=input_ids)

    @property
    def logit_scale(self):
        return self.hf.logit_scale


def load_clip_model(device=DEVICE):
    """Load OpenAI CLIP ViT-B/32, freeze every parameter, return
    (model, tokenize_fn, logit_scale, source_description)."""
    try:
        import open_clip
        model, _, _ = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRETRAINED)
        tokenize = open_clip.get_tokenizer(CLIP_ARCH)
        source = f"open_clip:{CLIP_ARCH}:{CLIP_PRETRAINED}"
    except Exception as e:  # pragma: no cover - fallback for restricted networks
        print(f"[load_clip_model] open_clip failed ({e}); falling back to HuggingFace transformers.")
        from transformers import CLIPModel, CLIPTokenizer
        tok = CLIPTokenizer.from_pretrained(CLIP_HF_FALLBACK)
        hf = CLIPModel.from_pretrained(CLIP_HF_FALLBACK)
        model = _HFTextImageWrapper(hf)
        tokenize = (lambda prompts: tok(prompts, padding="max_length", max_length=77,
                                        truncation=True, return_tensors="pt").input_ids)
        source = f"transformers:{CLIP_HF_FALLBACK}"

    for p in model.parameters():          # freeze the entire CLIP model
        p.requires_grad_(False)
    model = model.to(device).eval()
    with torch.no_grad():
        logit_scale = float(model.logit_scale.exp().item())
    print(f"[load_clip_model] source={source} | logit_scale={logit_scale:.2f} | device={device}")
    return model, tokenize, logit_scale, source


@torch.no_grad()
def extract_image_features(model, dataset, indices, device=DEVICE,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           use_amp=USE_AMP, desc=""):
    """Forward images through the FROZEN image encoder; return L2-normalized
    float32 features on CPU, shape [len(indices), 512]."""
    model.eval()
    idx_list = indices.tolist() if isinstance(indices, np.ndarray) else list(indices)
    loader = DataLoader(Subset(dataset, idx_list), batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device.type == "cuda"))
    feats = []
    for k, (images, _) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        if use_amp and device.type == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                f = model.encode_image(images)
        else:
            f = model.encode_image(images)
        feats.append(f.float().cpu())
        if desc and k % 20 == 0:
            print(f"  [{desc}] batch {k + 1}/{len(loader)}", flush=True)
    feats = torch.cat(feats, dim=0)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats


@torch.no_grad()
def encode_text_classes(model, tokenize, classes=CLASSES, templates=TEMPLATES, device=DEVICE):
    """Prompt-ensemble text embeddings: average of per-template normalized
    embeddings, re-normalized. Returns [num_classes, 512] float32 on CPU."""
    per_class = []
    for c in classes:
        prompts = [t.format(c) for t in templates]
        toks = tokenize(prompts).to(device)
        f = model.encode_text(toks).float()
        f = f / f.norm(dim=-1, keepdim=True)
        f = f.mean(dim=0)
        f = f / f.norm(dim=-1, keepdim=True)
        per_class.append(f.cpu())
    return torch.stack(per_class)


def build_feature_banks(model, tokenize, logit_scale, train_ds, test_ds, ytr, yte,
                        device=DEVICE, cache_dir=CACHE_DIR, mock=MOCK):
    """Extract (or load from disk cache) three feature banks shared by ALL methods:
         train_bank [50000, 512]  test_bank [10000, 512]  text_feats [10, 512]"""
    os.makedirs(cache_dir, exist_ok=True)
    paths = {k: os.path.join(cache_dir, f) for k, f in [
        ("train", "train_bank.pt"), ("test", "test_bank.pt"),
        ("text", "text_feats.pt"), ("scale", "logit_scale.pt")]}
    if all(os.path.exists(p) for p in paths.values()):
        print("[features] cached feature banks found on disk -- loading (skips image encoding)")
        return (torch.load(paths["train"]), torch.load(paths["test"]),
                torch.load(paths["text"]), float(torch.load(paths["scale"])))

    if mock:
        print("[features] MOCK mode: generating deterministic random feature banks")
        g = torch.Generator().manual_seed(7)
        train_bank = torch.randn(len(ytr), 512, generator=g)
        train_bank = train_bank / train_bank.norm(dim=-1, keepdim=True)
        test_bank = torch.randn(len(yte), 512, generator=g)
        test_bank = test_bank / test_bank.norm(dim=-1, keepdim=True)
        gt = torch.Generator().manual_seed(99)
        text_feats = torch.randn(NUM_CLASSES, 512, generator=gt)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        logit_scale = 100.0
    else:
        t0 = time.time()
        print(f"[features] encoding {len(ytr)} CIFAR-10 TRAIN images with frozen CLIP ...")
        train_bank = extract_image_features(model, train_ds, np.arange(len(ytr)),
                                            device=device, desc="train")
        print(f"[features]   train bank done in {time.time() - t0:.0f}s")
        t0 = time.time()
        print(f"[features] encoding {len(yte)} CIFAR-10 TEST images with frozen CLIP ...")
        test_bank = extract_image_features(model, test_ds, np.arange(len(yte)),
                                           device=device, desc="test")
        print(f"[features]   test bank done in {time.time() - t0:.0f}s")
        text_feats = encode_text_classes(model, tokenize, device=device)

    torch.save(train_bank, paths["train"])
    torch.save(test_bank, paths["test"])
    torch.save(text_feats, paths["text"])
    torch.save(torch.tensor(float(logit_scale)), paths["scale"])
    print(f"[features] banks cached to {cache_dir}")
    return train_bank, test_bank, text_feats, float(logit_scale)


## 11. The four methods
Zero-shot, linear probe, training-free Tip-Adapter (validation grid over
$\alpha \in \{0, 0.25, ..., 6\} \times \beta \in \{2,3,4,5,5.86,7,8\}$), and
Tip-Adapter-F (AdamW on the trainable keys, lr x wd grid, early stopping on validation,
post-training $\alpha$ re-search). Trainable-parameter verification is built into the
training function (`assert` that only `cache_keys` receives gradients).

In [ ]:
# =============================================================================
# methods.py -- the four adaptation methods
#
#   1) Zero-shot CLIP        : argmax over logit_scale * cos(img, class-text)
#   2) Linear Probe          : nn.Linear(512, 10) on frozen features, trained
#                              with cross-entropy on the support set
#   3) Tip-Adapter           : training-free cache model
#   4) Tip-Adapter-F         : same cache, but keys are nn.Parameter and
#                              optimized with AdamW + cross-entropy
#
# Tip-Adapter cache model (paper equations):
#   cache_keys K in R^(N x 512)  = L2-normalized support image features
#   cache_values V in {0,1}^(N x 10) = one-hot labels of the support images
#   affinity   A(q, k) = exp(-beta * (1 - cos(q, k)))          in (0, 1]
#   cache_logits(q)    = A @ V                                  [Q, 10]
#   final_logits(q)    = clip_logits(q) + alpha * cache_logits(q)
#
# Role of the two hyperparameters:
#   * beta -- sharpness of the affinity. Large beta -> only cache keys that are
#     nearly identical to the query contribute (nearest-neighbour behaviour);
#     beta -> 0 makes affinity nearly uniform (1 for every key).
#   * alpha -- global weight of the cache branch relative to CLIP's zero-shot
#     logits. alpha = 0 recovers pure zero-shot CLIP.
#
# Both are tuned on the VALIDATION set only, never on test accuracy.
# =============================================================================
import copy
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from config import (ALPHA_GRID, BETA_GRID, DEVICE, LP_EVAL_EVERY, LP_EPOCHS,
                        LP_LR_GRID, LP_WD_GRID, NUM_CLASSES, TIPF_LR_GRID,
                        TIPF_MAX_EPOCHS, TIPF_PATIENCE, TIPF_WD_GRID)
except ImportError:
    pass  # shared-namespace mode (notebook): names already defined by config cell


# ---------------------------------------------------------------------------
# Shared building blocks
# ---------------------------------------------------------------------------
def clip_logits(feats, text_feats, logit_scale):
    """CLIP zero-shot logits: logit_scale * cos(img, class_text).
    `feats` [B, 512] and `text_feats` [10, 512] must be L2-normalized."""
    return logit_scale * (feats @ text_feats.t())


def affinity_matrix(query_feats, cache_keys, beta=None, kind="exp"):
    """Affinity between query features and cache keys.
    kind='exp'    : exp(-beta * (1 - cos))  -- the Tip-Adapter affinity.
    kind='cosine' : raw cosine similarity    -- used only in ablation B.
    """
    sim = query_feats @ cache_keys.t()          # cosine similarity, [Q, N]
    if kind == "exp":
        return torch.exp(-beta * (1.0 - sim))
    if kind == "cosine":
        return sim
    raise ValueError(f"unknown affinity kind: {kind}")


def cache_logits(query_feats, cache_keys, cache_values, beta=None, kind="exp"):
    """cache_logits = affinity @ one-hot values -> [Q, 10] class scores."""
    return affinity_matrix(query_feats, cache_keys, beta, kind) @ cache_values


def one_hot_values(labels, num_classes=NUM_CLASSES):
    """cache_values: [N, num_classes] one-hot label matrix."""
    return F.one_hot(torch.as_tensor(labels, dtype=torch.long), num_classes).float()


def zero_shot_predict(feats, text_feats, logit_scale):
    logits = clip_logits(feats, text_feats, logit_scale)
    return logits, logits.argmax(dim=1)


# ---------------------------------------------------------------------------
# Method 2: linear probe on frozen CLIP features
# ---------------------------------------------------------------------------
def train_linear_probe(support_feats, support_labels, val_feats, val_labels,
                       num_classes=NUM_CLASSES, lr_grid=LP_LR_GRID, wd_grid=LP_WD_GRID,
                       epochs=LP_EPOCHS, eval_every=LP_EVAL_EVERY, device=DEVICE, seed=0,
                       verbose=True):
    """Full-batch linear head on frozen features. Grid-search (lr, weight_decay);
    for each config keep the epoch with the best validation accuracy. The best
    config is selected by validation accuracy (ties -> lower val loss)."""
    sf = support_feats.to(device)
    y = torch.as_tensor(support_labels, dtype=torch.long, device=device)
    vf = val_feats.to(device)
    vy = torch.as_tensor(val_labels, dtype=torch.long, device=device)

    best, t0 = None, time.time()
    for lr in lr_grid:
        for wd in wd_grid:
            torch.manual_seed(seed)
            head = nn.Linear(sf.shape[1], num_classes).to(device)
            opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
            best_ep, best_vacc, best_vloss, best_state = 0, -1.0, float("inf"), None
            for ep in range(1, epochs + 1):
                head.train()
                opt.zero_grad()
                loss = F.cross_entropy(head(sf), y)
                loss.backward()
                opt.step()
                if ep % eval_every == 0 or ep == epochs:
                    head.eval()
                    with torch.no_grad():
                        vl = head(vf)
                        vloss = F.cross_entropy(vl, vy).item()
                        vacc = (vl.argmax(1) == vy).float().mean().item()
                    if vacc > best_vacc or (vacc == best_vacc and vloss < best_vloss):
                        best_ep, best_vacc, best_vloss = ep, vacc, vloss
                        best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            if verbose:
                print(f"  [linear-probe] lr={lr:g} wd={wd:g} -> best val acc {best_vacc:.4f} @ epoch {best_ep}")
            if best is None or best_vacc > best["val_acc"] or \
                    (best_vacc == best["val_acc"] and best_vloss < best["val_loss"]):
                best = dict(state=best_state, lr=lr, wd=wd, val_acc=best_vacc,
                            val_loss=best_vloss, best_epoch=best_ep)
    train_time = time.time() - t0
    head = nn.Linear(sf.shape[1], num_classes).to(device)
    head.load_state_dict(best["state"])
    head.eval()
    return head, best, train_time


# ---------------------------------------------------------------------------
# Method 3: Tip-Adapter (training-free)
# ---------------------------------------------------------------------------
def tip_adapter_free_search(val_feats, val_labels, support_feats, support_values,
                            val_clip_logits, alpha_grid=ALPHA_GRID, beta_grid=None,
                            kind="exp", device=DEVICE, verbose=True, label="tip-adapter"):
    """Exhaustive (alpha, beta) grid search on the VALIDATION set.

    Returns (best_cfg dict with alpha/beta/val_acc, full sweep rows)."""
    if beta_grid is None:
        beta_grid = BETA_GRID
    vy = torch.as_tensor(val_labels, dtype=torch.long, device=device)
    sfd = support_feats.to(device)
    svd = support_values.to(device)
    vcl = val_clip_logits.to(device)
    sims_val = val_feats.to(device) @ sfd.t()      # [V, N] cosine similarities

    sweep, best = [], None
    for beta in beta_grid:
        if kind == "exp":
            cache_vl = torch.exp(-beta * (1.0 - sims_val)) @ svd
            alphas = alpha_grid
        else:  # raw cosine affinity: beta is irrelevant
            cache_vl = sims_val @ svd
            alphas = [a for a in alpha_grid if a > 0]
        for alpha in alphas:
            logits = vcl + alpha * cache_vl
            vacc = (logits.argmax(1) == vy).float().mean().item()
            sweep.append(dict(alpha=float(alpha), beta=float(beta), kind=kind, val_acc=vacc))
            if best is None or vacc > best["val_acc"]:
                best = dict(alpha=float(alpha), beta=float(beta), kind=kind, val_acc=vacc)
    if verbose:
        print(f"  [{label}] best ({kind}) on validation: alpha={best['alpha']:.2f} "
              f"beta={best['beta']:.2f} -> val acc {best['val_acc']:.4f}")
    return best, sweep


def tip_adapter_predict(test_feats, support_feats, support_values, text_feats,
                        logit_scale, alpha, beta, kind="exp", device=DEVICE):
    """final_logits = clip_logits + alpha * cache_logits on the test features."""
    tf = test_feats.to(device)
    logits = clip_logits(tf, text_feats.to(device), logit_scale) + \
        alpha * cache_logits(tf, support_feats.to(device), support_values.to(device),
                             beta, kind)
    return logits.cpu(), logits.argmax(1).cpu()


# ---------------------------------------------------------------------------
# Method 4: Tip-Adapter-F (fine-tuned cache keys)
# ---------------------------------------------------------------------------
class TipAdapterF(nn.Module):
    """Cache model with TRAINABLE keys.

    Only `cache_keys` is an nn.Parameter; cache_values / text_feats / logit_scale
    are registered buffers (constants). CLIP's encoders are NOT part of this
    module at all -- their logits are supplied precomputed and constant -- so it
    is impossible to update CLIP weights here by construction.

    Normalization: keys are initialized from L2-normalized support features and
    are NOT re-normalized during training (official implementation behaviour);
    queries are always L2-normalized frozen features."""

    def __init__(self, cache_keys, cache_values, text_feats, logit_scale, alpha, beta):
        super().__init__()
        self.cache_keys = nn.Parameter(cache_keys.clone())          # [N, 512] TRAINABLE
        self.register_buffer("cache_values", cache_values.clone())  # [N, 10] fixed
        self.register_buffer("text_feats", text_feats.clone())      # [10, 512] fixed
        self.register_buffer("logit_scale", torch.tensor(float(logit_scale)))
        self.alpha, self.beta = float(alpha), float(beta)

    def forward(self, query_feats):
        sim = query_feats @ self.cache_keys.t()
        cache_l = torch.exp(-self.beta * (1.0 - sim)) @ self.cache_values
        clip_l = self.logit_scale * (query_feats @ self.text_feats.t())
        return clip_l + self.alpha * cache_l

    def trainable_summary(self):
        names = [n for n, p in self.named_parameters() if p.requires_grad]
        count = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return names, count


def train_tip_adapter_f(support_feats, support_labels, support_values, val_feats, val_labels,
                        text_feats, logit_scale, alpha, beta, lr, wd,
                        max_epochs=TIPF_MAX_EPOCHS, patience=TIPF_PATIENCE,
                        device=DEVICE, seed=0):
    """One Tip-Adapter-F training run. Training queries ARE the support features
    (official protocol: the cache is optimized on the few-shot samples it stores).
    Cross-entropy on final_logits; AdamW; early stopping + checkpoint selection
    on VALIDATION accuracy (ties -> lower val loss)."""
    torch.manual_seed(seed)
    model = TipAdapterF(support_feats.to(device), support_values.to(device),
                        text_feats.to(device), logit_scale, alpha, beta).to(device)

    trainable_names, trainable_count = model.trainable_summary()
    assert trainable_names == ["cache_keys"], f"unexpected trainable params: {trainable_names}"
    assert trainable_count == np.prod(support_feats.shape), \
        f"expected {np.prod(support_feats.shape)} trainable values, got {trainable_count}"

    opt = torch.optim.AdamW([model.cache_keys], lr=lr, weight_decay=wd)
    sf = support_feats.to(device)
    y = torch.as_tensor(support_labels, dtype=torch.long, device=device)
    vf = val_feats.to(device)
    vy = torch.as_tensor(val_labels, dtype=torch.long, device=device)

    history = []
    best_vacc, best_vloss, best_epoch, best_state, wait = -1.0, float("inf"), 0, None, 0
    t0 = time.time()
    for ep in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(sf), y)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            vlog = model(vf)
            vloss = F.cross_entropy(vlog, vy).item()
            vacc = (vlog.argmax(1) == vy).float().mean().item()
        history.append(dict(epoch=ep, train_loss=float(loss.item()),
                            val_acc=vacc, val_loss=vloss))

        if vacc > best_vacc or (vacc == best_vacc and vloss < best_vloss):
            best_vacc, best_vloss, best_epoch, wait = vacc, vloss, ep, 0
            best_state = model.cache_keys.detach().clone()
        else:
            wait += 1
            if wait >= patience:
                break
    train_time = time.time() - t0
    return dict(best_keys=best_state, val_acc=best_vacc, val_loss=best_vloss,
                best_epoch=best_epoch, epochs_run=history[-1]["epoch"],
                history=history, train_time=train_time, trainable_count=trainable_count)


def run_tip_adapter_f(support_feats, support_labels, support_values, val_feats, val_labels,
                      text_feats, logit_scale, init_alpha, beta,
                      lr_grid=TIPF_LR_GRID, wd_grid=TIPF_WD_GRID, alpha_grid=ALPHA_GRID,
                      device=DEVICE, seed=0, verbose=True, ckpt_path=None):
    """Full Tip-Adapter-F protocol for one (shots, seed):
      1. grid-search (lr, weight_decay); each config early-stops on validation.
      2. re-search alpha on validation with the best frozen keys (official
         two-stage protocol). beta stays fixed at its validation-tuned value
         from the training-free stage.
    Returns the best result incl. final (keys, alpha, beta) and history."""
    best = None
    for lr in lr_grid:
        for wd in wd_grid:
            res = train_tip_adapter_f(support_feats, support_labels, support_values,
                                      val_feats, val_labels, text_feats, logit_scale,
                                      alpha=init_alpha, beta=beta, lr=lr, wd=wd,
                                      device=device, seed=seed)
            if verbose:
                print(f"  [tip-adapter-F] lr={lr:g} wd={wd:g} -> val acc {res['val_acc']:.4f} "
                      f"(best epoch {res['best_epoch']}/{res['epochs_run']})")
            if best is None or res["val_acc"] > best["val_acc"] or \
                    (res["val_acc"] == best["val_acc"] and res["val_loss"] < best["val_loss"]):
                res = dict(res)
                res.update(lr=lr, wd=wd)
                best = res

    # --- post-training alpha re-search on validation (keys frozen) ---
    keys = best["best_keys"].to(device)
    vals = support_values.to(device)
    vcl = clip_logits(val_feats.to(device), text_feats.to(device), logit_scale)
    sims = val_feats.to(device) @ keys.t()
    best_alpha = None
    for a in alpha_grid:
        logits = vcl + a * (torch.exp(-beta * (1.0 - sims)) @ vals)
        vacc = (logits.argmax(1) == torch.as_tensor(val_labels, dtype=torch.long,
                                                    device=device)).float().mean().item()
        if best_alpha is None or vacc > best_alpha[1]:
            best_alpha = (float(a), vacc)
    best["final_alpha"] = best_alpha[0]
    best["val_acc_after_alpha_search"] = best_alpha[1]
    if verbose:
        print(f"  [tip-adapter-F] post-training alpha search: alpha={best['final_alpha']:.2f} "
              f"-> val acc {best['val_acc_after_alpha_search']:.4f}")

    if ckpt_path is not None:
        torch.save(dict(cache_keys=best["best_keys"].cpu(), alpha=best["final_alpha"],
                        beta=float(beta), lr=best["lr"], wd=best["wd"],
                        init_alpha=float(init_alpha), best_epoch=best["best_epoch"],
                        val_acc=best["val_acc_after_alpha_search"]), ckpt_path)
    return best


def tip_f_predict(test_feats, cache_keys, cache_values, text_feats, logit_scale,
                  alpha, beta, device=DEVICE):
    """Test-time prediction with the fine-tuned keys and re-searched alpha."""
    tf = test_feats.to(device)
    logits = clip_logits(tf, text_feats.to(device), logit_scale) + \
        alpha * cache_logits(tf, cache_keys.to(device), cache_values.to(device), beta, "exp")
    return logits.cpu(), logits.argmax(1).cpu()


## 12. Metrics
Top-1 / Top-5 accuracy, macro precision / recall / F1, per-class accuracy and the
confusion matrix - computed from the saved logits, never re-tuned.

In [ ]:
# =============================================================================
# evaluation.py -- metrics, record keeping, result tables
# =============================================================================
import json
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

try:
    from config import NUM_CLASSES
except ImportError:
    pass  # shared-namespace mode (notebook): names already defined by config cell


def compute_metrics(y_true, y_pred, logits=None, num_classes=NUM_CLASSES):
    """Top-1, Top-5, macro precision/recall/F1, per-class accuracy, confusion."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    top1 = float((y_true == y_pred).mean())

    top5 = None
    if logits is not None:
        lg = logits if torch.is_tensor(logits) else torch.as_tensor(logits)
        k = min(5, num_classes)
        top5_idx = torch.topk(lg, k=k, dim=1).indices.numpy()
        top5 = float(np.mean([y_true[i] in top5_idx[i] for i in range(len(y_true))]))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(num_classes)), average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    per_class = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return dict(top1=top1, top5=top5,
                macro_precision=float(prec), macro_recall=float(rec), macro_f1=float(f1),
                per_class_accuracy=[float(x) for x in per_class],
                confusion_matrix=cm.astype(int).tolist())


def to_py(obj):
    """Recursively convert numpy/torch containers to plain Python for JSON."""
    if isinstance(obj, dict):
        return {k: to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_py(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if torch.is_tensor(obj):
        return obj.detach().cpu().tolist()
    return obj


def save_records(records, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_py(records), f, indent=2)


def summarize_records(records):
    """Flatten records into a DataFrame without the bulky per-class fields."""
    rows = [{k: (json.dumps(v) if k == "hp" and isinstance(v, dict) else v)
             for k, v in r.items()
             if k not in ("per_class_accuracy", "confusion_matrix", "history")}
            for r in records]
    return pd.DataFrame(rows)


def pivot_mean_std(df, metric="top1", index="method", columns="shots"):
    mean = df.pivot_table(index=index, columns=columns, values=metric, aggfunc="mean")
    std = df.pivot_table(index=index, columns=columns, values=metric, aggfunc="std")
    return mean, std


## 13. Ablations
Ablation B (affinity function) is computed fresh for every shot setting on seed 42;
ablations A and C are derived from the main records at summary time.

In [ ]:
# =============================================================================
# ablations.py
#
# A. Cache contribution  : zero-shot CLIP  vs  CLIP + cache (Tip-Adapter).
#    -> measures how much a non-parametric few-shot memory adds to zero-shot.
# B. Affinity function   : raw cosine similarity  vs  exp(-beta*(1-cos)).
#    -> measures the effect of the exponential kernel (aggregation identical).
# C. Fine-tuning effect  : fixed cache keys (Tip-Adapter) vs trainable keys
#    (Tip-Adapter-F) -> measures what gradient descent on the cache buys.
# D. Number of shots     : covered by the main 1/2/4/8/16-shot sweep.
#
# This file is executed into the shared namespace; functions from methods.py /
# data.py / config.py are already defined when it runs.
# =============================================================================
import numpy as np
import torch


def ablate_affinity(ctx, shots_list=None, seeds=(42,)):
    """Ablation B: for each shot setting, run the SAME (alpha, beta) validation
    search twice -- once with the exponential affinity and once with raw cosine
    similarity -- then evaluate both on the test set. Aggregation (affinity @
    one-hot values, added to CLIP logits) is identical in both arms."""
    if shots_list is None:
        shots_list = globals().get("SHOTS") or [1, 2, 4, 8, 16]
    train_bank, test_bank, text_feats = ctx["train_bank"], ctx["test_bank"], ctx["text_feats"]
    logit_scale, ytr, yte = ctx["logit_scale"], ctx["ytr"], ctx["yte"]
    test_clip_l = clip_logits(test_bank, text_feats, logit_scale)

    rows = []
    for shots in shots_list:
        for seed in seeds:
            support_idx, val_idx = make_few_shot_splits(ytr, shots, seed)
            support_feats = train_bank[support_idx]
            support_values = one_hot_values(ytr[support_idx], NUM_CLASSES)
            val_feats = train_bank[val_idx]
            val_labels = torch.as_tensor(ytr[val_idx], dtype=torch.long)
            val_clip_l = clip_logits(val_feats, text_feats, logit_scale)

            best_exp, _ = tip_adapter_free_search(val_feats, val_labels, support_feats,
                                                  support_values, val_clip_l,
                                                  kind="exp", verbose=False)
            best_cos, _ = tip_adapter_free_search(val_feats, val_labels, support_feats,
                                                  support_values, val_clip_l,
                                                  kind="cosine", verbose=False)
            exp_logits = test_clip_l + best_exp["alpha"] * cache_logits(
                test_bank, support_feats, support_values, best_exp["beta"], "exp")
            cos_logits = test_clip_l + best_cos["alpha"] * cache_logits(
                test_bank, support_feats, support_values, None, "cosine")
            rows.append(dict(
                shots=int(shots), seed=int(seed),
                exp_alpha=best_exp["alpha"], exp_beta=best_exp["beta"],
                exp_val=best_exp["val_acc"],
                exp_top1=float((exp_logits.argmax(1).numpy() == yte).mean()),
                cos_alpha=best_cos["alpha"], cos_val=best_cos["val_acc"],
                cos_top1=float((cos_logits.argmax(1).numpy() == yte).mean())))
    return rows


def derive_cache_and_finetune_table(df):
    """Ablations A and C, derived from the main experiment records:
       A: zero_shot -> tip_adapter        (adding the training-free cache)
       C: tip_adapter -> tip_adapter_f    (making the cache keys trainable)"""
    rows = []
    for shots in sorted(df["shots"].unique()):
        def mean_acc(method):
            sel = df[(df.method == method) & (df.shots == shots)]
            return float(sel["top1"].mean()) * 100
        rows.append(dict(
            shots=int(shots),
            zero_shot=mean_acc("zero_shot"),
            clip_plus_cache=mean_acc("tip_adapter"),
            clip_plus_trained_cache=mean_acc("tip_adapter_f"),
            ablationA_cache_gain=mean_acc("tip_adapter") - mean_acc("zero_shot"),
            ablationC_finetune_gain=mean_acc("tip_adapter_f") - mean_acc("tip_adapter")))
    return rows


## 14. Figures
All plots are saved as 200-dpi PNGs under `figures/`.

In [ ]:
# =============================================================================
# visualize.py -- publication-quality figures, all saved as PNG
# =============================================================================
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter, ScalarFormatter

try:
    from config import CLASSES, FIG_DIR, SEEDS, SHOTS
except ImportError:
    pass  # shared-namespace mode (notebook): names already defined by config cell

METHOD_ORDER = ["zero_shot", "linear_probe", "tip_adapter", "tip_adapter_f"]
METHOD_LABELS = {
    "zero_shot": "Zero-shot CLIP",
    "linear_probe": "Linear Probe",
    "tip_adapter": "Tip-Adapter (training-free)",
    "tip_adapter_f": "Tip-Adapter-F (fine-tuned)",
}
METHOD_COLORS = {
    "zero_shot": "#8c8c8c",
    "linear_probe": "#4477AA",
    "tip_adapter": "#EE7733",
    "tip_adapter_f": "#228833",
}


def _save(fig, out_dir, name):
    path = os.path.join(out_dir, name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  [figure] saved {path}")
    return path


def _shots_axis(ax):
    ax.set_xscale("log", base=2)
    ax.set_xticks(SHOTS)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("Shots (support images per class)")


def _pick(records, method, shots, seed):
    return next(r for r in records
                if r["method"] == method and r["shots"] == shots and r["seed"] == seed)


def plot_accuracy_vs_shots(df, out_dir):
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    for m in METHOD_ORDER:
        sub = df[df.method == m].groupby("shots")["top1"]
        shots = sorted(sub.groups.keys())
        mean = sub.mean().loc[shots] * 100
        std = sub.std().loc[shots].fillna(0) * 100
        ax.errorbar(shots, mean, yerr=std, marker="o", ms=5, capsize=3, lw=1.8,
                    color=METHOD_COLORS[m], label=METHOD_LABELS[m])
    _shots_axis(ax)
    ax.set_ylabel("Top-1 accuracy (%)")
    ax.set_title("Few-shot accuracy on CIFAR-10 (CLIP ViT-B/32)\n"
                 f"mean +/- std over seeds {SEEDS}")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=9)
    return _save(fig, out_dir, "01_accuracy_vs_shots.png")


def plot_tip_vs_tipf(df, out_dir):
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    x = np.arange(len(SHOTS))
    w = 0.38
    pairs = [("tip_adapter", "Tip-Adapter (training-free)"),
             ("tip_adapter_f", "Tip-Adapter-F (fine-tuned)")]
    for k, (m, label) in enumerate(pairs):
        sub = df[df.method == m].groupby("shots")["top1"]
        mean = sub.mean().loc[SHOTS] * 100
        std = sub.std().loc[SHOTS].fillna(0) * 100
        bars = ax.bar(x + (k - 0.5) * w, mean, w, yerr=std, capsize=3,
                      color=METHOD_COLORS[m], label=label)
        ax.bar_label(bars, fmt="%.1f", fontsize=8, padding=2)
    lo = df[df.method.isin(["tip_adapter", "tip_adapter_f"])]["top1"].min() * 100
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}-shot" for s in SHOTS])
    ax.set_ylabel("Top-1 accuracy (%)")
    ax.set_ylim(max(0.0, lo - 4), 100)
    ax.set_title("Tip-Adapter vs Tip-Adapter-F\n(error bars: std over seeds)")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=9)
    return _save(fig, out_dir, "02_tipadapter_vs_tipadapterf.png")


def plot_tipf_curves(records, out_dir, seed=None):
    seed = SEEDS[0] if seed is None else seed
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    colors = plt.cm.viridis(np.linspace(0.08, 0.85, len(SHOTS)))
    for ci, shots in enumerate(SHOTS):
        try:
            rec = _pick(records, "tip_adapter_f", shots, seed)
        except StopIteration:
            continue
        hist = rec.get("history")
        if not hist:
            continue
        ep = [h["epoch"] for h in hist]
        axes[0].plot(ep, [h["train_loss"] for h in hist], color=colors[ci],
                     marker=".", ms=4, label=f"{shots}-shot")
        axes[1].plot(ep, [h["val_acc"] * 100 for h in hist], color=colors[ci],
                     marker=".", ms=4, label=f"{shots}-shot")
        best_acc = hist[rec["best_epoch"] - 1]["val_acc"] * 100
        axes[1].scatter([rec["best_epoch"]], [best_acc], color=colors[ci], marker="*",
                        s=120, zorder=5, edgecolor="k", linewidth=0.4)
    axes[0].set_title(f"Tip-Adapter-F training loss (seed {seed})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy on support set")
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    axes[1].set_title("Validation accuracy per epoch (star = selected checkpoint)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation top-1 (%)")
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)
    return _save(fig, out_dir, "03_tipf_training_curves.png")


def plot_confusion_grid(records, out_dir, shots=16, seed=None):
    seed = SEEDS[0] if seed is None else seed
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 10.2))
    last_im = None
    for ax, m in zip(axes.ravel(), METHOD_ORDER):
        rec = _pick(records, m, shots, seed)
        cm = np.array(rec["confusion_matrix"])
        last_im = ax.imshow(cm, cmap="Blues")
        thr = cm.max() / 2.0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6.5,
                        color="white" if cm[i, j] > thr else "#333333")
        ax.set_title(f"{METHOD_LABELS[m]}\n{shots}-shot, seed {seed} -- top-1 {rec['top1'] * 100:.2f}%",
                     fontsize=10)
        ax.set_xticks(range(10))
        ax.set_yticks(range(10))
        ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(CLASSES, fontsize=7)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    if last_im is not None:
        fig.colorbar(last_im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.015)
    fig.suptitle(f"Confusion matrices on the CIFAR-10 test set (10,000 images, {shots}-shot, seed {seed})",
                 fontsize=12)
    return _save(fig, out_dir, "04_confusion_matrices_16shot.png")


def plot_per_class_accuracy(records, out_dir, shots=16, seed=None):
    seed = SEEDS[0] if seed is None else seed
    fig, ax = plt.subplots(figsize=(11, 4.8))
    x = np.arange(len(CLASSES))
    w = 0.2
    lows = []
    for k, m in enumerate(METHOD_ORDER):
        acc = np.array(_pick(records, m, shots, seed)["per_class_accuracy"]) * 100
        lows.append(acc.min())
        ax.bar(x + (k - 1.5) * w, acc, w, color=METHOD_COLORS[m], label=METHOD_LABELS[m])
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, rotation=30, ha="right")
    ax.set_ylabel("Per-class accuracy (%)")
    ax.set_ylim(max(0.0, min(lows) - 4), 101)
    ax.set_title(f"Per-class accuracy on the test set ({shots}-shot, seed {seed})")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=9, ncol=2)
    return _save(fig, out_dir, "05_per_class_accuracy_16shot.png")


def plot_training_time(records, out_dir):
    df = summarize_records(records)
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    x = np.arange(len(SHOTS))
    w = 0.38
    for k, m in enumerate(["linear_probe", "tip_adapter_f"]):
        sub = df[df.method == m].groupby("shots")["train_time_s"]
        mean = sub.mean().loc[SHOTS]
        std = sub.std().loc[SHOTS].fillna(0)
        ax.bar(x + (k - 0.5) * w, mean, w, yerr=std, capsize=3,
               color=METHOD_COLORS[m], label=METHOD_LABELS[m])
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}-shot" for s in SHOTS])
    ax.set_ylabel("Training time (s, mean over seeds)")
    ax.set_title("Training time: Linear Probe vs Tip-Adapter-F\n"
                 "(zero-shot and training-free Tip-Adapter need no training;\n"
                 "the Tip-Adapter (alpha, beta) search takes < 1 s per run)")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=9)
    return _save(fig, out_dir, "06_training_time.png")


def plot_trainable_params(records, out_dir):
    df = summarize_records(records)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    labels = ["Zero-shot CLIP", "Tip-Adapter"]
    vals = [0, 0]
    lp = int(df[df.method == "linear_probe"]["trainable_params"].max())
    labels.append("Linear Probe")
    vals.append(lp)
    for s in SHOTS:
        sub = df[(df.method == "tip_adapter_f") & (df.shots == s)]
        labels.append(f"Tip-Adapter-F ({s}-shot)")
        vals.append(int(sub["trainable_params"].max()))
    y = np.arange(len(labels))
    colors = ["#8c8c8c", "#EE7733", "#4477AA"] + ["#228833"] * len(SHOTS)
    ax.barh(y, vals, color=colors)
    for yi, v in zip(y, vals):
        ax.text(v + max(vals) * 0.015 + 400, yi, f"{v:,}" if v else "0 (training-free)",
                va="center", fontsize=8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Number of trainable parameters")
    ax.set_title("Trainable parameters per method\n"
                 "(CLIP image + text encoders, ~151M params, stay frozen in ALL methods)")
    ax.set_xlim(0, max(vals) * 1.2)
    ax.grid(alpha=0.3, axis="x")
    return _save(fig, out_dir, "07_trainable_params.png")


def plot_affinity_ablation(affinity_rows, out_dir):
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    x = np.arange(len(SHOTS))
    w = 0.38
    exp_m = [r["exp_top1"] * 100 for r in affinity_rows]
    cos_m = [r["cos_top1"] * 100 for r in affinity_rows]
    b1 = ax.bar(x - w / 2, cos_m, w, color="#8c8c8c", label="Raw cosine similarity")
    b2 = ax.bar(x + w / 2, exp_m, w, color="#228833",
                label=r"Exponential: exp($-\beta$(1 - cos))")
    ax.bar_label(b1, fmt="%.1f", fontsize=8)
    ax.bar_label(b2, fmt="%.1f", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}-shot" for s in SHOTS])
    ax.set_ylabel("Test top-1 accuracy (%)")
    ax.set_ylim(max(0.0, min(cos_m + exp_m) - 4), 100)
    ax.set_title("Ablation B: affinity function (seed 42;\nhyperparameters tuned on validation)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis="y")
    return _save(fig, out_dir, "08_ablation_affinity.png")


def plot_alpha_sensitivity(sweep, out_dir, seed=None):
    seed = SEEDS[0] if seed is None else seed
    if not sweep:
        return None
    sw = pd.DataFrame(sweep)
    fig, ax = plt.subplots(figsize=(8, 5))
    for beta, g in sw.groupby("beta"):
        g = g.sort_values("alpha")
        ax.plot(g["alpha"], g["val_acc"] * 100, marker=".", ms=4, label=f"beta = {beta:g}")
    ax.set_xlabel("alpha  (weight of the cache logits)")
    ax.set_ylabel("Validation top-1 (%)")
    ax.set_title(f"Role of alpha and beta (training-free Tip-Adapter, 16-shot, seed {seed})\n"
                 "alpha scales the cache contribution; beta sharpens nearest-neighbour affinity")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    return _save(fig, out_dir, "09_alpha_beta_sensitivity.png")


def make_all_figures(records, affinity_rows, sweep_16, out_dir=FIG_DIR):
    os.makedirs(out_dir, exist_ok=True)
    df = summarize_records(records)
    plot_accuracy_vs_shots(df, out_dir)
    plot_tip_vs_tipf(df, out_dir)
    plot_tipf_curves(records, out_dir)
    plot_confusion_grid(records, out_dir)
    plot_per_class_accuracy(records, out_dir)
    plot_training_time(records, out_dir)
    plot_trainable_params(records, out_dir)
    if affinity_rows:
        plot_affinity_ablation(affinity_rows, out_dir)
    if sweep_16:
        plot_alpha_sensitivity(sweep_16, out_dir)


## 15. Run the experiment
One call executes: feature banks -> 60 evaluated runs (4 methods x 5 shots x 3 seeds)
-> ablations -> 9 figures -> summary tables. Records are saved incrementally, so the
run is resumable at the artifacts level if interrupted.

In [ ]:
# =============================================================================
# run_experiments.py -- end-to-end orchestration
#
#   Stage 1  data + frozen CLIP feature banks (cached on disk)
#   Stage 2  4 methods x 5 shot settings x 3 seeds = 60 evaluated runs
#   Stage 3  ablations (affinity function; cache contribution; fine-tuning)
#   Stage 4  figures (PNG)
#   Stage 5  summary tables (CSV + console)
#
# Shared-namespace loader: locally this file execs the other modules into its
# own globals; inside the Kaggle notebook the same code is placed in cells, the
# files are absent, and the loader is a no-op.
# =============================================================================
import json
import os
import time

_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()


def _load_shared(name):
    path = os.path.join(_HERE, name)
    if os.path.exists(path):
        exec(compile(open(path, encoding="utf-8").read(), path, "exec"), globals())
    else:
        print(f"[run_experiments] {name} not on disk -- assuming notebook mode (names already defined).")


for _name in ("config.py", "data.py", "clip_features.py", "methods.py",
              "evaluation.py", "ablations.py", "visualize.py"):
    _load_shared(_name)

import numpy as np
import torch

# NOTE: after the exec loop above, every function/constant from the project
# modules is available as a plain global (shared-namespace design), so no
# project imports are needed here.


def set_all_seeds(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_environment():
    print("=" * 78)
    print("ENVIRONMENT")
    print("=" * 78)
    print(f"device: {DEVICE} | torch {torch.__version__} | mock mode: {MOCK}")
    if DEVICE.type == "cuda":
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name} | VRAM: {p.total_memory / 1e9:.1f} GB")
    try:
        import torchvision
        print(f"torchvision {torchvision.__version__}")
    except Exception:
        pass
    try:
        import open_clip
        print(f"open_clip {open_clip.__version__}")
    except Exception:
        print("open_clip: not installed (transformers fallback will be used)")


def stage_data_and_model():
    print("\n" + "=" * 78)
    print("STAGE 1 -- DATA + FROZEN CLIP FEATURE BANKS")
    print("=" * 78)
    ensure_dirs()
    set_all_seeds(SEEDS[0])
    train_ds, test_ds, ytr, yte = load_datasets()
    print(f"CIFAR-10: train {len(ytr)} | test {len(yte)} | classes {len(CLASSES)}")

    clip_source, img_params = "mock", 0
    if MOCK:
        model, tokenize, logit_scale = None, None, 100.0
    else:
        model, tokenize, logit_scale, clip_source = load_clip_model()
        named = [(n, p) for n, p in model.named_parameters()]
        img_params = sum(p.numel() for n, p in named if n.startswith("visual."))
        total = sum(p.numel() for p in model.parameters())
        frozen = all(not p.requires_grad for p in model.parameters())
        print(f"CLIP parameters: total {total:,} | all frozen (requires_grad=False): {frozen}")
        if img_params:
            print(f"  image encoder ~{img_params:,} | text encoder ~{total - img_params:,}")

    train_bank, test_bank, text_feats, logit_scale = build_feature_banks(
        model, tokenize, logit_scale, train_ds, test_ds, ytr, yte)
    for nm, m in (("train", train_bank), ("test", test_bank), ("text", text_feats)):
        err = float((m.norm(dim=-1) - 1).abs().max())
        assert err < 1e-4, f"{nm} bank rows are not unit-normalized (max err {err})"
    print(f"train bank {tuple(train_bank.shape)} | test bank {tuple(test_bank.shape)} | "
          f"text {tuple(text_feats.shape)} | logit_scale {logit_scale:.2f}")

    if not MOCK:
        print(f"CLIP frozen check (all requires_grad=False): "
              f"{all(not p.requires_grad for p in model.parameters())}")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Explicit verification of what is trainable in Tip-Adapter-F (spec item)
    demo_keys = torch.randn(16, train_bank.shape[1])
    demo_vals = one_hot_values([i % NUM_CLASSES for i in range(16)], NUM_CLASSES)
    demo = TipAdapterF(demo_keys, demo_vals, text_feats, logit_scale, alpha=1.0, beta=5.86)
    tnames, tcount = demo.trainable_summary()
    print(f"Tip-Adapter-F trainable parameter names: {tnames}")
    print(f"Tip-Adapter-F trainable parameter count (demo, 16 cache keys): "
          f"{tcount} = 16 x {train_bank.shape[1]}")
    print("Everything else (cache values, text embeddings, both CLIP encoders) "
          "is frozen by construction.")
    return dict(train_bank=train_bank, test_bank=test_bank, text_feats=text_feats,
                logit_scale=logit_scale, ytr=ytr, yte=yte, clip_source=clip_source)


def dump_run_config(ctx):
    cfg = dict(seeds=SEEDS, shots=SHOTS, val_per_class=VAL_PER_CLASS,
               num_classes=NUM_CLASSES, classes=CLASSES, templates=TEMPLATES,
               alpha_grid=ALPHA_GRID, beta_grid=BETA_GRID,
               lp_lr_grid=LP_LR_GRID, lp_wd_grid=LP_WD_GRID, lp_epochs=LP_EPOCHS,
               tipf_lr_grid=TIPF_LR_GRID, tipf_wd_grid=TIPF_WD_GRID,
               tipf_max_epochs=TIPF_MAX_EPOCHS, tipf_patience=TIPF_PATIENCE,
               clip_source=ctx["clip_source"], logit_scale=ctx["logit_scale"],
               feature_dim=int(ctx["train_bank"].shape[1]),
               torch=torch.__version__, mock=bool(MOCK))
    try:
        import torchvision
        cfg["torchvision"] = torchvision.__version__
    except Exception:
        pass
    try:
        import open_clip
        cfg["open_clip"] = open_clip.__version__
    except Exception:
        pass
    with open(os.path.join(RESULTS_DIR, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    print(f"[config] run configuration saved to {os.path.join(RESULTS_DIR, 'run_config.json')}")


def make_record(method, shots, seed, metrics, hp=None, train_time_s=0.0,
                trainable_params=0, epochs_run=0, best_epoch=0, val_top1=None,
                history=None):
    rec = dict(method=method, shots=int(shots), seed=int(seed),
               top1=metrics["top1"], top5=metrics["top5"],
               macro_precision=metrics["macro_precision"],
               macro_recall=metrics["macro_recall"], macro_f1=metrics["macro_f1"],
               per_class_accuracy=metrics["per_class_accuracy"],
               confusion_matrix=metrics["confusion_matrix"],
               support_size=int(shots) * NUM_CLASSES,
               train_time_s=float(train_time_s), trainable_params=int(trainable_params),
               epochs_run=int(epochs_run), best_epoch=int(best_epoch),
               val_top1=None if val_top1 is None else float(val_top1),
               hp=hp or {})
    if history is not None:
        rec["history"] = history
    return rec


def run_one(shots, seed, ctx, verbose=True):
    """One (shots, seed) run: all four methods on the SAME support samples."""
    train_bank, test_bank, text_feats = ctx["train_bank"], ctx["test_bank"], ctx["text_feats"]
    logit_scale, ytr, yte = ctx["logit_scale"], ctx["ytr"], ctx["yte"]

    support_idx, val_idx = make_few_shot_splits(ytr, shots, seed)
    support_feats = train_bank[support_idx]
    support_labels = ytr[support_idx]
    support_values = one_hot_values(support_labels, NUM_CLASSES)
    val_feats, val_labels = train_bank[val_idx], ytr[val_idx]
    test_feats, test_labels = test_bank, yte

    test_clip_l = clip_logits(test_feats, text_feats, logit_scale)
    val_clip_l = clip_logits(val_feats, text_feats, logit_scale)
    n_support, dim = support_feats.shape
    if verbose:
        print(f"\n--- shots={shots} seed={seed} | support={n_support} images "
              f"({shots}/class) | val={len(val_idx)} images ---")
    recs = []

    # ---- 1) Zero-shot CLIP ---------------------------------------------------
    t0 = time.time()
    zs_logits, zs_pred = zero_shot_predict(test_feats, text_feats, logit_scale)
    zs_time = time.time() - t0
    m = compute_metrics(test_labels, zs_pred.numpy(), zs_logits.numpy())
    recs.append(make_record("zero_shot", shots, seed, m, hp={}, train_time_s=zs_time))
    if verbose:
        print(f"  zero-shot     : test top1 {m['top1'] * 100:.2f}% | trainable params 0")

    # ---- 2) Linear probe -------------------------------------------------------
    head, lp_best, lp_time = train_linear_probe(
        support_feats, torch.as_tensor(support_labels), val_feats,
        torch.as_tensor(val_labels), seed=seed, verbose=False)
    with torch.no_grad():
        lp_logits = head(test_feats.to(DEVICE)).cpu()
    m = compute_metrics(test_labels, lp_logits.argmax(1).numpy(), lp_logits.numpy())
    lp_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
    recs.append(make_record("linear_probe", shots, seed, m,
                            hp=dict(lr=lp_best["lr"], wd=lp_best["wd"]),
                            train_time_s=lp_time, trainable_params=lp_params,
                            epochs_run=LP_EPOCHS, best_epoch=lp_best["best_epoch"],
                            val_top1=lp_best["val_acc"]))
    if verbose:
        print(f"  linear-probe  : test top1 {m['top1'] * 100:.2f}% | val {lp_best['val_acc'] * 100:.2f}% "
              f"| trainable params {lp_params} | time {lp_time:.1f}s")

    # ---- 3) Tip-Adapter (training-free) ---------------------------------------
    t0 = time.time()
    best_free, sweep = tip_adapter_free_search(
        val_feats, torch.as_tensor(val_labels), support_feats, support_values,
        val_clip_l, verbose=verbose)
    search_time = time.time() - t0
    tip_logits_test = test_clip_l + best_free["alpha"] * cache_logits(
        test_feats, support_feats, support_values, best_free["beta"])
    m = compute_metrics(test_labels, tip_logits_test.argmax(1).numpy(), tip_logits_test.numpy())
    recs.append(make_record("tip_adapter", shots, seed, m, hp=best_free,
                            train_time_s=search_time, val_top1=best_free["val_acc"]))
    if verbose:
        print(f"  tip-adapter   : test top1 {m['top1'] * 100:.2f}% | alpha={best_free['alpha']:.2f} "
              f"beta={best_free['beta']:.2f} | trainable params 0 | search {search_time:.2f}s")
    sweep_save = sweep if (shots == 16 and seed == SEEDS[0]) else None

    # ---- 4) Tip-Adapter-F -------------------------------------------------------
    ckpt = os.path.join(CKPT_DIR, f"tip_adapter_f_{shots}shot_seed{seed}.pt")
    init_alpha = max(float(best_free["alpha"]), 0.5)   # guard: keep gradient flowing to keys
    res = run_tip_adapter_f(
        support_feats, torch.as_tensor(support_labels), support_values, val_feats,
        torch.as_tensor(val_labels), text_feats, logit_scale,
        init_alpha=init_alpha, beta=float(best_free["beta"]), seed=seed, ckpt_path=ckpt)
    tf_logits, tf_pred = tip_f_predict(test_feats, res["best_keys"], support_values,
                                       text_feats, logit_scale, res["final_alpha"],
                                       float(best_free["beta"]))
    m = compute_metrics(test_labels, tf_pred.numpy(), tf_logits.numpy())
    tf_params = n_support * dim
    recs.append(make_record("tip_adapter_f", shots, seed, m,
                            hp=dict(lr=res["lr"], wd=res["wd"], init_alpha=init_alpha,
                                    beta=float(best_free["beta"]), final_alpha=res["final_alpha"],
                                    max_epochs=TIPF_MAX_EPOCHS, patience=TIPF_PATIENCE),
                            train_time_s=res["train_time"], trainable_params=tf_params,
                            epochs_run=res["epochs_run"], best_epoch=res["best_epoch"],
                            val_top1=res["val_acc_after_alpha_search"],
                            history=res["history"]))
    if verbose:
        print(f"  tip-adapter-F : test top1 {m['top1'] * 100:.2f}% | val {res['val_acc_after_alpha_search'] * 100:.2f}% "
              f"| lr={res['lr']:g} wd={res['wd']:g} final_alpha={res['final_alpha']:.2f} "
              f"| best epoch {res['best_epoch']}/{res['epochs_run']} | time {res['train_time']:.1f}s "
              f"| trainable params {tf_params} ({n_support} keys x {dim})")
    return recs, sweep_save


def stage_experiments(ctx):
    print("\n" + "=" * 78)
    print("STAGE 2 -- FEW-SHOT EXPERIMENTS (4 methods x 5 shots x 3 seeds)")
    print("=" * 78)
    records, sweep_16 = [], None
    total = len(SHOTS) * len(SEEDS)
    done, t_start = 0, time.time()
    for shots in SHOTS:
        for seed in SEEDS:
            recs, sweep = run_one(shots, seed, ctx)
            records.extend(recs)
            if sweep is not None:
                sweep_16 = sweep
                save_records(sweep, os.path.join(RESULTS_DIR, "alpha_beta_sweep_16shot_seed42.json"))
            done += 1
            save_records(records, os.path.join(RESULTS_DIR, "records.json"))
            print(f"[progress] {done}/{total} runs done | elapsed {time.time() - t_start:.0f}s", flush=True)
    return records


def stage_ablations(ctx):
    print("\n" + "=" * 78)
    print("STAGE 3 -- ABLATIONS")
    print("=" * 78)
    rows = ablate_affinity(ctx)
    save_records(rows, os.path.join(RESULTS_DIR, "ablation_affinity.json"))
    for r in rows:
        print(f"  {r['shots']:>2}-shot | exp  (alpha={r['exp_alpha']:.2f}, beta={r['exp_beta']:.2f}) "
              f"-> test {r['exp_top1'] * 100:.2f}%   |   cosine (alpha={r['cos_alpha']:.2f}) "
              f"-> test {r['cos_top1'] * 100:.2f}%")
    return rows


def stage_visualize(records, affinity_rows, sweep_16):
    print("\n" + "=" * 78)
    print("STAGE 4 -- FIGURES")
    print("=" * 78)
    make_all_figures(records, affinity_rows, sweep_16, FIG_DIR)


def stage_summary(records, affinity_rows):
    print("\n" + "=" * 78)
    print("STAGE 5 -- SUMMARY TABLES")
    print("=" * 78)
    df = summarize_records(records)
    df.to_csv(os.path.join(RESULTS_DIR, "records.csv"), index=False)

    order = [m for m in ("zero_shot", "linear_probe", "tip_adapter", "tip_adapter_f")]
    mean, std = pivot_mean_std(df, "top1", "method", "shots")
    mean, std = mean.loc[order], std.loc[order]

    print(f"\nTop-1 accuracy (%) on the official CIFAR-10 test set -- mean +/- std "
          f"over seeds {SEEDS}")
    tbl = pd.DataFrame(index=mean.index, columns=mean.columns, dtype=object)
    for c in mean.columns:
        tbl[c] = [f"{m * 100:.2f} +/- {s * 100:.2f}"
                  for m, s in zip(mean[c], std[c].fillna(0))]
    tbl.index = [METHOD_LABELS[i] for i in tbl.index]
    print(tbl.to_string())
    tbl.to_csv(os.path.join(RESULTS_DIR, "summary_top1_mean_std.csv"))

    d16 = df[df.shots == 16]
    show = d16.groupby("method")[["top1", "top5", "macro_precision", "macro_recall",
                                  "macro_f1"]].mean().loc[order] * 100
    print("\nFull test metrics at 16-shot (mean over seeds, %):")
    print(show.round(2).to_string())
    aux = d16.groupby("method")[["trainable_params", "train_time_s"]].mean().loc[order]
    print("\nTrainable parameters / training time at 16-shot (mean over seeds):")
    print(aux.round(2).to_string())

    cache_tbl = pd.DataFrame(derive_cache_and_finetune_table(df))
    cache_tbl.to_csv(os.path.join(RESULTS_DIR, "ablation_cache_and_finetune.csv"), index=False)
    print("\nAblation A (cache contribution) and C (fine-tuning effect), test top-1 %:")
    print(cache_tbl.round(2).to_string(index=False))

    aff = pd.DataFrame(affinity_rows)
    print("\nAblation B (affinity function), seed 42, test top-1 %:")
    print(aff[["shots", "cos_top1", "exp_top1"]].assign(
        cos_top1=lambda d: d.cos_top1 * 100, exp_top1=lambda d: d.exp_top1 * 100
    ).round(2).to_string(index=False))


def list_outputs():
    print("\nOutput files:")
    for base in (RESULTS_DIR, FIG_DIR, CKPT_DIR):
        if not os.path.isdir(base):
            continue
        for root, _, files in os.walk(base):
            for fn in sorted(files):
                p = os.path.join(root, fn)
                try:
                    print(f"  {p} ({os.path.getsize(p) / 1e6:.2f} MB)")
                except OSError:
                    pass


def run_all():
    t0 = time.time()
    print_environment()
    ctx = stage_data_and_model()
    dump_run_config(ctx)
    records = stage_experiments(ctx)
    affinity_rows = stage_ablations(ctx)
    sweep_path = os.path.join(RESULTS_DIR, "alpha_beta_sweep_16shot_seed42.json")
    sweep_16 = json.load(open(sweep_path, encoding="utf-8")) if os.path.exists(sweep_path) else None
    stage_visualize(records, affinity_rows, sweep_16)
    stage_summary(records, affinity_rows)
    print(f"\nTotal wall-clock: {(time.time() - t0) / 60:.1f} min")
    list_outputs()
    print("\nALL DONE")
    return records


In [ ]:
records = run_all()

## 16. Reading the results
* `results/records.json` / `records.csv` -- one row per (method, shots, seed) with all
  metrics, timings, trainable-parameter counts and chosen hyperparameters.
* `results/summary_top1_mean_std.csv` -- the headline mean +/- std table.
* `figures/*.png` -- the publication figures.
* `checkpoints/*.pt` -- best fine-tuned cache keys per run.
* `results/run_config.json` -- exact configuration + library versions for reproducibility.
* `feature_cache/*.pt` -- cached frozen CLIP embeddings (skip re-extraction on re-runs).
